# Point Transformer for 3D Shape Classification

3D Shape Classification on ModelNet40: Vector self-attention transformer for 3D geometric point set classification. This notebook implements the approach with `PointTransformerConv` inside a `K3PointTransformer` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `PointTransformerConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Point Transformer for 3D Shape Classification"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Point Transformer Model Definition
class K3PointTransformer(keras.Model):
    def __init__(self, in_channels, out_channels=10):
        super().__init__()
        self.conv1 = k3_layers.PointTransformerConv(in_channels, 32)
        self.conv2 = k3_layers.PointTransformerConv(32, 64)
        self.lin = layers.Dense(out_channels)

    def call(self, pos, edge_index, batch=None):
        x = ops.relu(self.conv1(pos, pos, edge_index))
        x = ops.relu(self.conv2(x, pos, edge_index))
        out = k3_layers.global_max_pool(x, batch)
        return self.lin(out)

k3_model = K3PointTransformer(in_channels=3, out_channels=10)

# 2. Forward pass test
num_points = 64
dummy_pos = keras.random.normal((num_points, 3))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")
dummy_batch = ops.zeros((num_points,), dtype="int64")

out = k3_model(dummy_pos, dummy_edges, dummy_batch)
print(f"PointTransformer forward pass successful! Output shape: {out.shape}")

print("\n✓ K3-Node Point Transformer Classification execution completed successfully!")